# 10b — Weight Initialization (PyTorch)

Corresponds to: Note 10 — Weight Initialization

**Libraries introduced in this notebook:** No new libraries — `torch` and `torch.nn` introduced in Note 09.

**What this notebook demonstrates:**
1. He and Xavier initialization via `torch.nn.init` — compared to the manual formulas from notebook 10a
2. Activation variance trace — same 8-layer experiment as 10a, using `nn.Module` and PyTorch hooks
3. Symmetry — zero vs He initialization using `nn.Linear`, gradients verified after one backward pass

In [ ]:
import torch
import torch.nn as nn

print(f'PyTorch version: {torch.__version__}')

## Part 1: He and Xavier Initialization via `torch.nn.init`

In notebook 10a we computed σ manually:

    Xavier:  σ = √(2 / (n_in + n_out))
    He:      σ = √(2 / n_in)

PyTorch provides these as `nn.init.xavier_normal_` and `nn.init.kaiming_normal_`.
Both operate in-place on a weight tensor.

We apply them to the same [64, 32, 16, 1] architecture from 10a and verify the actual std
matches the formula.

In [ ]:
torch.manual_seed(42)

layer_sizes = [64, 32, 16, 1]

print(f"{'Layer':<8} {'n_in':<8} {'n_out':<8} {'Formula σ':<12} {'Actual σ':<12} {'Method'}")
print("-" * 62)

for l in range(1, len(layer_sizes)):
    n_in  = layer_sizes[l - 1]
    n_out = layer_sizes[l]
    linear = nn.Linear(n_in, n_out)

    if l < len(layer_sizes) - 1:
        # He init for ReLU hidden layers
        nn.init.kaiming_normal_(linear.weight, mode='fan_in', nonlinearity='relu')
        formula_std = (2 / n_in) ** 0.5
        method = 'He (kaiming_normal_)'
    else:
        # Xavier init for linear output layer
        nn.init.xavier_normal_(linear.weight)
        formula_std = (2 / (n_in + n_out)) ** 0.5
        method = 'Xavier (xavier_normal_)'

    actual_std = linear.weight.std().item()
    print(f"{l:<8} {n_in:<8} {n_out:<8} {formula_std:<12.4f} {actual_std:<12.4f} {method}")

The actual std of each initialized layer matches the formula from Note 10 closely.

In practice, initialization is applied inside the model definition. The standard pattern
is to call `apply` with a custom init function after building the model.

In [ ]:
def init_weights(module):
    """Apply He init to Linear layers with ReLU, Xavier to the output layer."""
    if isinstance(module, nn.Linear):
        nn.init.kaiming_normal_(module.weight, mode='fan_in', nonlinearity='relu')
        nn.init.zeros_(module.bias)

torch.manual_seed(0)
model = nn.Sequential(
    nn.Linear(64, 32), nn.ReLU(),
    nn.Linear(32, 16), nn.ReLU(),
    nn.Linear(16,  1)
)
model.apply(init_weights)

print('Weight std after init_weights:')
for name, param in model.named_parameters():
    if 'weight' in name:
        print(f'  {name:<20}  std = {param.std().item():.4f}')

## Part 2: Activation Variance Trace

In 10a we ran a forward pass through an 8-layer numpy network and recorded activation std per layer.
Here we do the same in PyTorch using `nn.Module` and forward hooks to capture intermediate activations.

A forward hook is a function PyTorch calls automatically after each layer's forward pass —
it receives the layer's output and lets us record it without modifying the model.

In [ ]:
def build_deep_network(n_units, n_layers, init_fn):
    """Build an n_layers ReLU network, all layers n_units wide, apply init_fn."""
    layers = []
    for _ in range(n_layers):
        layers.append(nn.Linear(n_units, n_units))
        layers.append(nn.ReLU())
    net = nn.Sequential(*layers)
    net.apply(init_fn)
    return net

def activation_stds(net, x):
    """Run forward pass, return std of output after each ReLU layer."""
    stds = []
    hooks = []

    def hook_fn(module, input, output):
        stds.append(output.std().item())

    for module in net.modules():
        if isinstance(module, nn.ReLU):
            hooks.append(module.register_forward_hook(hook_fn))

    with torch.no_grad():
        net(x)

    for h in hooks:
        h.remove()

    return stds

torch.manual_seed(42)
n_units   = 256
n_layers  = 8
n_samples = 1000
x = torch.randn(n_samples, n_units)

def init_too_small(m):
    if isinstance(m, nn.Linear):
        nn.init.normal_(m.weight, std=0.01)
        nn.init.zeros_(m.bias)

def init_too_large(m):
    if isinstance(m, nn.Linear):
        nn.init.normal_(m.weight, std=1.0)
        nn.init.zeros_(m.bias)

def init_he(m):
    if isinstance(m, nn.Linear):
        nn.init.kaiming_normal_(m.weight, mode='fan_in', nonlinearity='relu')
        nn.init.zeros_(m.bias)

def init_xavier(m):
    if isinstance(m, nn.Linear):
        nn.init.xavier_normal_(m.weight)
        nn.init.zeros_(m.bias)

configs = [
    ('too_small', init_too_small),
    ('too_large', init_too_large),
    ('he',        init_he),
    ('xavier',    init_xavier),
]

results = {}
for name, init_fn in configs:
    torch.manual_seed(0)
    net = build_deep_network(n_units, n_layers, init_fn)
    results[name] = activation_stds(net, x)

print(f"{'Layer':<8}" + "".join(f"{name:<14}" for name, _ in configs))
print("-" * 64)
for l in range(n_layers):
    row = f"{l+1:<8}"
    for name, _ in configs:
        row += f"{results[name][l]:<14.4f}"
    print(row)

The result matches 10a: too-small init collapses activations to near zero, too-large causes explosion,
He initialization keeps std stable across all 8 layers.

## Part 3: Symmetry — Zero vs He Initialization

In 10a we proved algebraically and demonstrated numerically that zero initialization causes
all neurons in a layer to receive identical gradients. Here we reproduce the same demonstration
using `nn.Linear` and PyTorch autograd.

Network: 2 inputs → 2 hidden neurons → 1 output, sigmoid activations, BCE loss.
We run one forward + backward pass and compare the gradients of the two hidden neurons.

In [ ]:
x      = torch.tensor([[0.5, 0.8]])
y_true = torch.tensor([[1.0]])

def one_pass(init_fn, label):
    torch.manual_seed(0)
    model = nn.Sequential(
        nn.Linear(2, 2),
        nn.Sigmoid(),
        nn.Linear(2, 1),
        nn.Sigmoid()
    )
    init_fn(model)

    y_hat = model(x)
    loss  = nn.BCELoss()(y_hat, y_true)
    loss.backward()

    # Gradient of loss w.r.t. weights of the first Linear layer
    grad = model[0].weight.grad   # shape (2, 2): [neuron0_grads, neuron1_grads]
    print(f"{label}:")
    print(f"  Gradient neuron 0: {grad[0].tolist()}")
    print(f"  Gradient neuron 1: {grad[1].tolist()}")
    print(f"  Gradients identical: {torch.allclose(grad[0], grad[1])}")
    print()

def zero_init(model):
    for m in model.modules():
        if isinstance(m, nn.Linear):
            nn.init.zeros_(m.weight)
            nn.init.zeros_(m.bias)

def he_init(model):
    for m in model.modules():
        if isinstance(m, nn.Linear):
            nn.init.kaiming_normal_(m.weight, mode='fan_in', nonlinearity='relu')
            nn.init.zeros_(m.bias)

one_pass(zero_init, 'Zero initialization')
one_pass(he_init,   'He initialization')

## Summary

This notebook demonstrated:

1. **He and Xavier via `torch.nn.init`** — `nn.init.kaiming_normal_` and `nn.init.xavier_normal_` produce the same σ as the manual formulas from 10a. The standard pattern is `model.apply(init_fn)` after building the model.
2. **Activation variance trace** — using forward hooks to capture intermediate activations without modifying the model. Results match 10a: too-small init collapses, too-large explodes, He keeps std stable across 8 layers.
3. **Symmetry** — with zero initialization, PyTorch autograd produces identical gradients for both hidden neurons after one backward pass, confirming the algebraic result from Note 10 Part 2. He initialization breaks symmetry immediately.